# 04 - Feature Engineering (FIXED)

This is a corrected version of `04_feature_engineering.ipynb`.

**Bug 1 fixed — row-count explosion:** the original version merged
`product_data` / `seller_data` back onto `master_df` using
`merge(..., on="order_id")` while both sides still had multiple rows
per `order_id` (one per line item). Merging on a non-unique key on both
sides produces a many-to-many join — for every order with N items it
multiplied that order's rows by N again. That's what took the dataset
from 112,650 rows to 1,898,230 rows.

**Bug 2 fixed — item-level counting instead of distinct-order counting:**
even after fixing the row-count bug, `product_order_count` /
`seller_order_count` were still computed with a plain
`groupby(...).cumcount()` on item-level rows. If the same product (or
seller) appeared twice within one order, that counted as two separate
"previous orders" instead of one. Fixed by deduping to one row per
`(product_id, order_id)` / `(seller_id, order_id)` pair — aggregating
same-product-same-order items into a single record — before computing
any cumulative count or historical average, then merging back with a
safe many-to-one join.

Other fixes in this version vs. the original:
- Removed duplicated cells (date features and delivery-delay features
  were computed twice in the original).
- Added IQR-based outlier capping for `price` and `freight_value`
  *before* any derived features are built from them, so outliers don't
  propagate into every downstream ratio/average.
- The target is decided explicitly (see the markdown cell before
  saving) and is persisted to disk — the original computed `y` in
  memory and never saved it.
- Row-count checks now assert (not just print) against
  `ORIGINAL_ROW_COUNT`, captured once from the loaded data rather than
  hard-coded, per the requested validation discipline.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

## Load the master dataset (output of notebook 03)

In [2]:
MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "master_dataset.csv"
master_df = pd.read_csv(MASTER_PATH)

# Store the true starting row count once -- every check below verifies
# against this instead of a hard-coded number, since the raw dataset
# size can change (e.g. a Kaggle version bump) without the *logic*
# of "feature engineering must not change row count" changing.
ORIGINAL_ROW_COUNT = len(master_df)

print("Loaded master_dataset.csv")
print("Rows:", ORIGINAL_ROW_COUNT, "| Columns:", master_df.shape[1])
print("Unique orders:", master_df["order_id"].nunique())

Loaded master_dataset.csv
Rows: 112650 | Columns: 29
Unique orders: 98666


In [3]:
def check_rowcount(df, step_name, expected=ORIGINAL_ROW_COUNT):
    """Row-count sanity check. Every feature block below must leave the
    row count untouched -- this is exactly the assertion style notebook
    03 used at every merge, applied here to every feature step.

    Uses a plain `assert` (not just a print) so a silent mismatch can
    never slip through a full "run all" pass -- this is the exact bug
    class that took the dataset from 112,650 to 1,898,230 rows before.
    """
    actual = len(df)
    print(f"[{'OK' if actual == expected else 'MISMATCH'}] {step_name}: "
          f"{actual:,} rows (expected {expected:,})")
    assert actual == expected, (
        f"Row count changed during '{step_name}': "
        f"{expected:,} -> {actual:,}. Stop and investigate before continuing."
    )

## 1. Datetime conversion (single pass)

In [4]:
datetime_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
]

for col in datetime_columns:
    if col in master_df.columns:
        master_df[col] = pd.to_datetime(master_df[col], errors="coerce")

check_rowcount(master_df, "datetime conversion")

[OK] datetime conversion: 112,650 rows (expected 112,650)


## 2. Calendar features

In [5]:
master_df["order_year"] = master_df["order_purchase_timestamp"].dt.year
master_df["order_month"] = master_df["order_purchase_timestamp"].dt.month
master_df["order_day"] = master_df["order_purchase_timestamp"].dt.day
master_df["order_day_of_week"] = master_df["order_purchase_timestamp"].dt.dayofweek
master_df["order_hour"] = master_df["order_purchase_timestamp"].dt.hour
master_df["order_week"] = master_df["order_purchase_timestamp"].dt.isocalendar().week.astype(int)
master_df["is_weekend"] = (master_df["order_day_of_week"] >= 5).astype(int)

check_rowcount(master_df, "calendar features")

[OK] calendar features: 112,650 rows (expected 112,650)


## 3. Delivery timing features (kept for a possible future logistics side-project; not used as the pricing model target)

In [6]:
master_df["delivery_days"] = (
    master_df["order_delivered_customer_date"] - master_df["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

master_df["estimated_delivery_days"] = (
    master_df["order_estimated_delivery_date"] - master_df["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

master_df["delivery_delay_days"] = (
    master_df["order_delivered_customer_date"] - master_df["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

master_df["approval_delay_hours"] = (
    master_df["order_approved_at"] - master_df["order_purchase_timestamp"]
).dt.total_seconds() / 3600

master_df["shipping_delay_days"] = (
    master_df["shipping_limit_date"] - master_df["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

master_df["is_delayed"] = (master_df["delivery_delay_days"] > 0).astype(int)

check_rowcount(master_df, "delivery timing features")
print(master_df["is_delayed"].value_counts(normalize=True))

[OK] delivery timing features: 112,650 rows (expected 112,650)
is_delayed
0    0.922636
1    0.077364
Name: proportion, dtype: float64


## 4. Outlier handling on `price` / `freight_value`

Done *before* any derived pricing features so the outliers don't leak
into averages, ratios, or logs computed later. Capping (winsorizing)
is used instead of dropping rows, so row count and order coverage are
preserved.

In [7]:
def iqr_cap(series, k=3.0):
    """Cap values outside [Q1 - k*IQR, Q3 + k*IQR]. k=3.0 is deliberately
    wide (vs. the usual 1.5) since price data is right-skewed by design
    (premium products are legitimately expensive, not erroneous)."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    return series.clip(lower=max(lower, 0), upper=upper), (max(lower, 0), upper)

for col in ["price", "freight_value"]:
    before_mean, before_max = master_df[col].mean(), master_df[col].max()
    master_df[col], bounds = iqr_cap(master_df[col])
    after_mean, after_max = master_df[col].mean(), master_df[col].max()
    print(f"{col}: capped to [{bounds[0]:.2f}, {bounds[1]:.2f}]")
    print(f"  before -> mean {before_mean:.2f}, max {before_max:.2f}")
    print(f"  after  -> mean {after_mean:.2f}, max {after_max:.2f}")

check_rowcount(master_df, "outlier capping")

price: capped to [0.00, 419.90]
  before -> mean 120.65, max 6735.00
  after  -> mean 105.83, max 419.90
freight_value: capped to [0.00, 45.36]
  before -> mean 19.99, max 409.68
  after  -> mean 18.69, max 45.36
[OK] outlier capping: 112,650 rows (expected 112,650)


## 5. Price / revenue base features

In [8]:
master_df["item_total_value"] = master_df["price"] + master_df["freight_value"]

master_df["freight_percentage"] = (
    master_df["freight_value"] / master_df["price"].replace(0, np.nan)
) * 100

master_df["total_payment_value"] = master_df["total_payment_value"].fillna(0)
master_df["payment_count"] = master_df["payment_count"].fillna(0)

master_df["payment_price_difference"] = (
    master_df["total_payment_value"] - master_df["item_total_value"]
)
master_df["payment_price_ratio"] = (
    master_df["total_payment_value"] / master_df["price"].replace(0, np.nan)
)

master_df["price_log"] = np.log1p(master_df["price"])
master_df["freight_log"] = np.log1p(master_df["freight_value"])

master_df["total_order_value"] = master_df["price"] + master_df["freight_value"]

check_rowcount(master_df, "price/revenue base features")

[OK] price/revenue base features: 112,650 rows (expected 112,650)


## 6. Historical / panel features

Two safe patterns are used below, chosen per feature:

- **Order-level features (customer, state/city, previous-month):**
  dedupe to one row per `order_id` first, then merge back on `order_id`.
  Safe because the deduped side has a unique key -- many-to-one, never
  many-to-many.
- **Product / seller features:** dedupe to one row per
  **(product_id or seller_id, order_id)** pair first -- aggregating
  same-product-same-order line items into a single record -- then merge
  back on that composite key. This is required (not just for row-count
  safety) because a product/seller can appear more than once within one
  order, and a plain item-level `cumcount()` would count that as
  multiple "previous orders" instead of one. Deduping to distinct orders
  first makes `product_order_count` / `seller_order_count` mean exactly
  "distinct previous orders", not "previous line items".

Both patterns are many-to-one merges, never many-to-many, so they can't
multiply rows -- the class of bug that broke the original notebook.
`check_rowcount` still runs after every block as a second line of
defense, backed by a real `assert`.

### 6.1 Customer historical features (order-level: dedupe on `order_id` first, then merge is safe since it's one-to-many)

In [9]:
customer_data = (
    master_df[["order_id", "customer_unique_id", "order_purchase_timestamp",
               "total_order_value", "price"]]
    .drop_duplicates("order_id")
    .sort_values(["customer_unique_id", "order_purchase_timestamp", "order_id"])
    .copy()
)

customer_data["customer_order_count"] = customer_data.groupby("customer_unique_id").cumcount()
customer_data["customer_total_spend"] = (
    customer_data.groupby("customer_unique_id")["total_order_value"]
    .transform(lambda x: x.shift(1).fillna(0).cumsum())
)
customer_data["customer_avg_order_value"] = (
    customer_data["customer_total_spend"] / customer_data["customer_order_count"].replace(0, np.nan)
).fillna(0)
customer_data["customer_avg_price"] = (
    customer_data.groupby("customer_unique_id")["price"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)
customer_data["is_repeat_customer"] = (customer_data["customer_order_count"] > 0).astype(int)

cust_cols = ["customer_order_count", "customer_total_spend", "customer_avg_order_value",
             "customer_avg_price", "is_repeat_customer"]

master_df = master_df.drop(columns=cust_cols, errors="ignore")
master_df = master_df.merge(
    customer_data[["order_id"] + cust_cols], on="order_id", how="left"
)

check_rowcount(master_df, "customer historical features")

[OK] customer historical features: 112,650 rows (expected 112,650)


### 6.2 Product historical features (deduped to one row per DISTINCT order so counts/revenue/price reflect distinct previous orders, not previous line items)

In [10]:
# FIX: the previous version grouped on item-level rows directly, so a
# product appearing twice in the SAME order (two line items) counted
# as two separate "previous" events instead of one order. Deduping to
# one row per (product_id, order_id) first -- aggregating multi-item
# same-order/same-product rows into a single order-level record --
# makes every downstream cumcount/shift genuinely mean "distinct
# previous orders", per the fix requested.

product_orders = (
    master_df
    .groupby(["product_id", "order_id"], as_index=False)
    .agg(
        order_purchase_timestamp=("order_purchase_timestamp", "min"),
        order_total_value=("total_order_value", "sum"),   # this order's total for this product
        order_avg_price=("price", "mean"),
        order_avg_freight=("freight_value", "mean"),
    )
    .sort_values(["product_id", "order_purchase_timestamp", "order_id"])
)

# Distinct previous orders containing this product (current order excluded).
product_orders["product_order_count"] = product_orders.groupby("product_id").cumcount()

# Historical revenue / avg price / avg freight computed over distinct
# previous orders only -- shift(1) excludes the current order, exactly
# as before, just now on the correctly-deduped order-level table.
product_orders["product_total_revenue"] = (
    product_orders.groupby("product_id")["order_total_value"]
    .transform(lambda x: x.shift(1).fillna(0).cumsum())
)
product_orders["product_avg_price"] = (
    product_orders.groupby("product_id")["order_avg_price"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)
product_orders["product_avg_freight"] = (
    product_orders.groupby("product_id")["order_avg_freight"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)

prod_cols = ["product_order_count", "product_total_revenue", "product_avg_price", "product_avg_freight"]

# Safe merge: product_orders has exactly one row per (product_id, order_id),
# so every master_df row -- even multiple line items of the same product
# in the same order -- matches exactly one row here. Many-to-one, never
# many-to-many, so this cannot multiply rows.
master_df = master_df.drop(columns=prod_cols, errors="ignore")
master_df = master_df.merge(
    product_orders[["product_id", "order_id"] + prod_cols],
    on=["product_id", "order_id"], how="left"
)

master_df["product_popularity"] = master_df["product_order_count"]

check_rowcount(master_df, "product historical features")
print("Unique orders after product-feature merge:", master_df["order_id"].nunique())

[OK] product historical features: 112,650 rows (expected 112,650)
Unique orders after product-feature merge: 98666


### 6.3 Seller historical features (same distinct-order fix as product features)

In [11]:
# Same fix as the product block above, same reasoning: dedupe to one
# row per (seller_id, order_id) before computing any historical count
# or average, so a seller with multiple items in the same order is
# counted as ONE previous order, not one per item.

seller_orders = (
    master_df
    .groupby(["seller_id", "order_id"], as_index=False)
    .agg(
        order_purchase_timestamp=("order_purchase_timestamp", "min"),
        order_total_value=("total_order_value", "sum"),
        order_avg_price=("price", "mean"),
        order_avg_freight=("freight_value", "mean"),
    )
    .sort_values(["seller_id", "order_purchase_timestamp", "order_id"])
)

seller_orders["seller_order_count"] = seller_orders.groupby("seller_id").cumcount()
seller_orders["seller_total_revenue"] = (
    seller_orders.groupby("seller_id")["order_total_value"]
    .transform(lambda x: x.shift(1).fillna(0).cumsum())
)
seller_orders["seller_avg_price"] = (
    seller_orders.groupby("seller_id")["order_avg_price"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)
seller_orders["seller_avg_freight"] = (
    seller_orders.groupby("seller_id")["order_avg_freight"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)

seller_cols = ["seller_order_count", "seller_total_revenue", "seller_avg_price", "seller_avg_freight"]

# Same safe many-to-one merge as the product block.
master_df = master_df.drop(columns=seller_cols, errors="ignore")
master_df = master_df.merge(
    seller_orders[["seller_id", "order_id"] + seller_cols],
    on=["seller_id", "order_id"], how="left"
)

check_rowcount(master_df, "seller historical features")
print("Unique orders after seller-feature merge:", master_df["order_id"].nunique())

[OK] seller historical features: 112,650 rows (expected 112,650)
Unique orders after seller-feature merge: 98666


### 6.4 Category historical order count (item-level)

In [12]:
tmp = (
    master_df[["order_id", "order_item_id", "product_category_name", "order_purchase_timestamp"]]
    .sort_values(["product_category_name", "order_purchase_timestamp", "order_id", "order_item_id"])
    .copy()
)
tmp["category_order_count"] = tmp.groupby("product_category_name").cumcount()

master_df = master_df.drop(columns=["category_order_count"], errors="ignore")
master_df.loc[tmp.index, "category_order_count"] = tmp["category_order_count"]

check_rowcount(master_df, "category historical features")

[OK] category historical features: 112,650 rows (expected 112,650)


### 6.5 State / city historical order counts (order-level: dedupe on `order_id` first)

In [13]:
location_data = (
    master_df[["order_id", "customer_state", "customer_city", "order_purchase_timestamp",
               "total_order_value"]]
    .drop_duplicates("order_id")
    .sort_values(["order_purchase_timestamp", "order_id"])
    .copy()
)

location_data["state_order_count"] = location_data.groupby("customer_state").cumcount()
location_data["city_order_count"] = location_data.groupby("customer_city").cumcount()
location_data["state_total_revenue"] = (
    location_data.sort_values(["customer_state", "order_purchase_timestamp", "order_id"])
    .groupby("customer_state")["total_order_value"]
    .transform(lambda x: x.shift(1).fillna(0).cumsum())
)
location_data["state_avg_order_value"] = (
    location_data["state_total_revenue"] / location_data["state_order_count"].replace(0, np.nan)
).fillna(0)

loc_cols = ["state_order_count", "city_order_count", "state_total_revenue", "state_avg_order_value"]
master_df = master_df.drop(columns=loc_cols, errors="ignore")
master_df = master_df.merge(location_data[["order_id"] + loc_cols], on="order_id", how="left")

check_rowcount(master_df, "state/city historical features")

[OK] state/city historical features: 112,650 rows (expected 112,650)


### 6.6 Previous-month demand features (order-level)

In [14]:
monthly_data = (
    master_df[["order_id", "order_purchase_timestamp", "order_year", "order_month",
               "total_order_value", "price"]]
    .drop_duplicates("order_id")
    .copy()
)

monthly_summary = (
    monthly_data.groupby(["order_year", "order_month"], as_index=False)
    .agg(monthly_orders=("order_id", "count"),
         monthly_revenue=("total_order_value", "sum"),
         monthly_avg_price=("price", "mean"))
    .sort_values(["order_year", "order_month"])
)
monthly_summary["previous_month_orders"] = monthly_summary["monthly_orders"].shift(1)
monthly_summary["previous_month_revenue"] = monthly_summary["monthly_revenue"].shift(1)
monthly_summary["previous_month_avg_price"] = monthly_summary["monthly_avg_price"].shift(1)

prev_cols = ["previous_month_orders", "previous_month_revenue", "previous_month_avg_price"]
master_df = master_df.drop(columns=prev_cols, errors="ignore")
master_df = master_df.merge(
    monthly_summary[["order_year", "order_month"] + prev_cols],
    on=["order_year", "order_month"], how="left"
)
for col in prev_cols:
    master_df[col] = master_df[col].fillna(0)

check_rowcount(master_df, "previous-month demand features")

[OK] previous-month demand features: 112,650 rows (expected 112,650)


## 7. Demand level (tercile bucket of product order count)

In [15]:
master_df["demand_level"] = pd.qcut(
    master_df["product_order_count"], q=3, labels=["Low", "Medium", "High"], duplicates="drop"
)
print(master_df["demand_level"].value_counts())
check_rowcount(master_df, "demand level")

demand_level
Low       51361
High      35456
Medium    25833
Name: count, dtype: int64
[OK] demand level: 112,650 rows (expected 112,650)


## 8. Final missing-value pass

In [16]:
numeric_columns = master_df.select_dtypes(include=["int64", "float64"]).columns
for col in numeric_columns:
    master_df[col] = master_df[col].fillna(master_df[col].median())

categorical_columns = master_df.select_dtypes(include=["object"]).columns
for col in categorical_columns:
    master_df[col] = master_df[col].fillna("Unknown")

print("Total missing values:", master_df.isnull().sum().sum())
check_rowcount(master_df, "final missing-value pass")

Total missing values: 3663
[OK] final missing-value pass: 112,650 rows (expected 112,650)


## 9. What this feature set is for, and a leakage audit

**Corrected note:** an earlier draft of this notebook said `price` and
`demand_level` would be the targets for notebook 05. That's outdated —
notebook 05 is **demand forecasting** (Prophet/XGBoost on daily order
volume), per the project's actual Phase 5. `price` and `demand_level`
stay in the saved file as engineered columns (useful for a future
Phase-6-style pricing notebook), but nothing in this pipeline currently
trains on them as a target.

**Leakage audit of the historical/panel features (customer, product,
seller, category, state/city, previous-month):** every one of them is
built the same way — sort chronologically, then `cumcount()` /
`.shift(1)` — which by construction excludes the current order and
only looks backward in time. None of them can see a future order.

**Known, deliberate exception — not a bug:** the IQR outlier caps on
`price`/`freight_value` (section 4) and the `demand_level` tercile
cutoffs (section 7) are computed from quantiles over the *entire*
dataset, including orders that happen after any given row. Strictly,
that's a mild form of look-ahead (the cutoffs reflect the full
population, not just "history up to this order"). It's a common,
generally-accepted simplification for outlier capping and static
binning, but it's flagged here explicitly rather than silently — a
stricter version would recompute these thresholds on an expanding
window per date, which is more complexity than this stage needs.

**Not a leakage risk in practice:** `is_delayed`, `delivery_days`,
`delivery_delay_days` etc. are post-outcome (only knowable after
delivery). They're kept as columns for a possible future logistics
notebook but are **not** used as predictors of anything in this
pipeline, and `05_demand_forecasting.ipynb` doesn't touch any
order-level attribute at all — it only uses the daily aggregate count
plus calendar/lag features derived from that same aggregate, so none
of the order-level leakage questions apply to it.

## 10. Drop identifiers / raw datetimes (not usable as model features) and save

In [17]:
# Final row-count / unique-order validation, captured before dropping
# order_id (needed here, no longer available on ml_df after this cell).
print("Final master_df shape:", master_df.shape)
print("Unique orders:", master_df["order_id"].nunique())
assert len(master_df) == ORIGINAL_ROW_COUNT, "Row count drifted before final save -- stop."

id_columns = ["order_id", "customer_id", "customer_unique_id", "order_item_id",
              "product_id", "seller_id"]
datetime_cols_to_drop = ["order_purchase_timestamp", "order_approved_at",
                          "order_delivered_carrier_date", "order_delivered_customer_date",
                          "order_estimated_delivery_date", "shipping_limit_date"]

ml_df = master_df.drop(columns=id_columns + datetime_cols_to_drop, errors="ignore")

check_rowcount(ml_df, "final ml dataset (post ID/datetime drop)")
print("Final ml_df shape:", ml_df.shape)
print(ml_df.dtypes.value_counts())

Final master_df shape: (112650, 72)
Unique orders: 98666
[OK] final ml dataset (post ID/datetime drop): 112,650 rows (expected 112,650)
Final ml_df shape: (112650, 60)
float64     39
int64       11
int32        5
object       4
category     1
Name: count, dtype: int64


In [18]:
FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "feature_engineered_dataset.csv"
ml_df.to_csv(FEATURE_PATH, index=False)
print("Saved:", FEATURE_PATH)
print("Rows:", ml_df.shape[0], "(should equal the notebook-03 row count of", ORIGINAL_ROW_COUNT, ")")

Saved: d:\pricing-optimization-system\data\processed\feature_engineered_dataset.csv
Rows: 112650 (should equal the notebook-03 row count of 112650 )
